# 💬 제조 AI 챗봇 (Manufacturing Chatbot)

> **학습 목표**: RAG 기반 제조 AI 대화형 챗봇 구축

## 📋 학습 내용
1. ✅ RAG 시스템 통합 (ChromaDB 활용)
2. ✅ 대화 메모리 관리 (Conversation Memory)
3. ✅ LangChain 체인 구축
4. ✅ Gradio 대화형 UI
5. ✅ 제조 도메인 맞춤화

**소요 시간**: 45분 | **난이도**: ⭐⭐⭐⭐ | **Part**: 3-1

## 🔧 Step 1: 라이브러리 Import 및 환경 설정

In [ ]:
# 필수 라이브러리
import warnings
warnings.filterwarnings('ignore')

import os
import json
from pathlib import Path
from datetime import datetime

# LangChain 라이브러리
from langchain.chains import ConversationalRetrievalChain
from langchain.memory import ConversationBufferMemory
from langchain.prompts import PromptTemplate
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import Chroma
from langchain.llms.fake import FakeListLLM

# UI 라이브러리
import gradio as gr

# 데이터 처리
import pandas as pd
import numpy as np

# 시각화
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# 한글 폰트 설정
plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False

print("✅ 라이브러리 Import 완료")
print(f"📅 실행 시간: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

## 📂 Step 2: RAG 시스템 로드 (ChromaDB)

In [ ]:
# ============================================================
# KAMP 실제 데이터 기반 RAG 시스템 로드
# 우선순위: 1) 기존 ChromaDB  2) KAMP PDF  3) KAMP JSON  4) 데모 문서
# ============================================================

from langchain.docstore.document import Document

# 경로 설정
chroma_db_dir = Path('../outputs/chroma_db')
kamp_data_dir = Path('../../dataset/part3-1')
kamp_pdf_files = [
    kamp_data_dir / '가이드북_시지트로닉스.pdf',
    kamp_data_dir / '가이드북_인스틸(주).pdf',
    kamp_data_dir / '가이드북_(주)성보산업.pdf',
]
kamp_json_dir = kamp_data_dir / '게시물'
kamp_csv_files = {
    '진부_공개용데이터': kamp_data_dir / '진부_공개용데이터_V1.csv',
    '에스케이지_제조AI': kamp_data_dir / '제조AI데이터셋_에스케이지 주식회사.csv',
    'Pump_Train': kamp_data_dir / 'Pump_Train_data_공개용.csv',
}

data_source = None

# 임베딩 모델 로드 (공통)
embeddings = HuggingFaceEmbeddings(
    model_name='jhgan/ko-sroberta-multitask',
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)

# ---- 1) 기존 ChromaDB 로드 시도 ----
if chroma_db_dir.exists():
    try:
        vectorstore = Chroma(
            persist_directory=str(chroma_db_dir),
            embedding_function=embeddings,
            collection_name='manufacturing_docs'
        )
        count = vectorstore._collection.count()
        if count > 0:
            data_source = f'ChromaDB (기존, {count}개 문서)'
            print(f"✅ 기존 ChromaDB 로드 완료: {count}개 문서")
    except Exception as e:
        print(f"⚠️ 기존 ChromaDB 로드 실패: {e}")
        data_source = None

# ---- 2) KAMP PDF 기반 새 ChromaDB 생성 ----
if data_source is None:
    kamp_docs = []
    
    try:
        from langchain.document_loaders import PyPDFLoader
        from langchain.text_splitter import RecursiveCharacterTextSplitter
        
        existing_pdfs = [p for p in kamp_pdf_files if p.exists()]
        if existing_pdfs:
            print(f"📂 KAMP 가이드북 PDF에서 RAG 구축: {len(existing_pdfs)}개")
            for pdf_path in existing_pdfs:
                try:
                    loader = PyPDFLoader(str(pdf_path))
                    pdf_docs = loader.load()
                    kamp_docs.extend(pdf_docs)
                    print(f"   ✅ {pdf_path.name}: {len(pdf_docs)} 페이지")
                except Exception as e:
                    print(f"   ⚠️ {pdf_path.name} 실패: {e}")
    except ImportError:
        print("⚠️ PyPDFLoader 미설치 (pip install pypdf)")
    
    # ---- 3) KAMP JSON 기반 문서 생성 ----
    if not kamp_docs and kamp_json_dir.exists():
        json_files = sorted(kamp_json_dir.glob('*.json'))
        if json_files:
            print(f"📂 KAMP 게시물 JSON에서 RAG 구축: {len(json_files)}개")
            for jf in json_files:
                try:
                    with open(jf, 'r', encoding='utf-8') as f:
                        data = json.load(f)
                    
                    company = data.get('업체명', '')
                    title = data.get('제목', '')
                    overview = data.get('제조AI데이터셋개요', {})
                    collection_info = data.get('제조AI데이터셋수집방법', {})
                    
                    content = f"""# {title}
업체명: {company}
수집업종: {data.get('수집업종', '')}
수집목적: {data.get('수집목적', '')}
수집공정설비: {data.get('수집공정설비', '')}

## 구축 배경
{overview.get('제조데이터셋 구축 배경', '')}

## 수집 방법
{overview.get('제조데이터셋 수집 방법', '')}

## 주요 변수
{overview.get('제조데이터셋 주요 변수', '')}

## 활용 방법
{overview.get('제조데이터셋 활용 방법', '')}
"""
                    kamp_docs.append(Document(
                        page_content=content,
                        metadata={'source': jf.name, 'company': company, 'category': data.get('수집업종', '')}
                    ))
                except Exception as e:
                    print(f"   ⚠️ {jf.name}: {e}")
    
    # KAMP 문서가 있으면 ChromaDB 생성
    if kamp_docs:
        text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=200)
        chunks = text_splitter.split_documents(kamp_docs)
        
        chroma_db_dir.mkdir(parents=True, exist_ok=True)
        vectorstore = Chroma.from_documents(
            documents=chunks,
            embedding=embeddings,
            persist_directory=str(chroma_db_dir),
            collection_name='manufacturing_docs'
        )
        data_source = f'KAMP 데이터 ({len(kamp_docs)}개 문서, {len(chunks)}개 청크)'
        print(f"✅ KAMP 기반 ChromaDB 생성 완료: {len(chunks)}개 청크")

# ---- 4) Fallback: 데모 문서 ----
if data_source is None:
    print("⚠️ KAMP 데이터 없음. 데모용 RAG 시스템을 생성합니다...")
    
    demo_docs = [
        Document(
            page_content="예지보전은 설비 고장을 사전에 예측하여 유지보수 비용을 절감합니다. 진동 센서와 온도 센서 데이터를 분석하여 이상 징후를 탐지합니다.",
            metadata={'source': 'demo_predictive_maintenance', 'category': '예지보전'}
        ),
        Document(
            page_content="품질 관리 시스템은 불량품을 자동으로 분류하고 원인을 분석합니다. 비전 AI를 활용하여 긁힘, 오염, 파손 등을 실시간으로 탐지합니다.",
            metadata={'source': 'demo_quality_control', 'category': '품질관리'}
        ),
        Document(
            page_content="스마트 팩토리는 IoT 센서, AI, 빅데이터를 통합하여 생산성을 향상시킵니다. 실시간 모니터링과 자동화로 생산 효율을 최대화합니다.",
            metadata={'source': 'demo_smart_factory', 'category': '스마트팩토리'}
        ),
        Document(
            page_content="설비 고장 진단은 머신러닝 모델을 사용하여 고장 유형을 분류합니다. Random Forest와 XGBoost 모델이 높은 정확도를 보입니다.",
            metadata={'source': 'demo_fault_diagnosis', 'category': '고장진단'}
        )
    ]
    
    chroma_db_dir.mkdir(parents=True, exist_ok=True)
    vectorstore = Chroma.from_documents(
        documents=demo_docs,
        embedding=embeddings,
        persist_directory=str(chroma_db_dir),
        collection_name='manufacturing_docs'
    )
    data_source = '데모 문서 (4개)'
    print("✅ 데모용 RAG 시스템 생성 완료")

# KAMP CSV 데이터 경로 표시 (데이터 분석 도구용)
print(f"\n📊 KAMP CSV 데이터셋:")
for name, path in kamp_csv_files.items():
    status = "✅ 존재" if path.exists() else "❌ 없음"
    size = f"({path.stat().st_size / 1024 / 1024:.1f}MB)" if path.exists() else ""
    print(f"   - {name}: {status} {size}")

# Retriever 설정
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={'k': 3}
)

print(f"\n📊 데이터 소스: {data_source}")
print(f"📁 ChromaDB 경로: {chroma_db_dir}")

## 🧠 Step 3: 대화 메모리 설정 (Conversation Memory)

In [ ]:
# ConversationBufferMemory 설정
memory = ConversationBufferMemory(
    memory_key='chat_history',
    return_messages=True,
    output_key='answer'
)

print("✅ 대화 메모리 설정 완료")
print("💡 ConversationBufferMemory: 전체 대화 내역을 저장")
print("   - memory_key: 'chat_history'")
print("   - return_messages: True (메시지 객체 반환)")
print("   - output_key: 'answer' (응답 키 지정)")

## 🤖 Step 4: LLM 및 프롬프트 설정

In [ ]:
# 제조 도메인 맞춤 프롬프트
manufacturing_prompt_template = """당신은 제조 분야 AI 전문가입니다. 
아래 문서를 참고하여 사용자의 질문에 답변하세요.

**답변 가이드라인**:
1. 제공된 문서 내용을 기반으로 정확하게 답변하세요
2. 문서에 없는 내용은 "제공된 문서에서는 해당 정보를 찾을 수 없습니다"라고 답변하세요
3. 기술적인 용어는 쉽게 풀어서 설명하세요
4. 실무에 적용 가능한 구체적인 조언을 제공하세요

참고 문서:
{context}

대화 기록:
{chat_history}

질문: {question}

답변:"""

MANUFACTURING_PROMPT = PromptTemplate(
    template=manufacturing_prompt_template,
    input_variables=['context', 'chat_history', 'question']
)

# FakeListLLM 설정 (데모용 - 실제 환경에서는 OpenAI API 사용)
# 실제 사용 시: from langchain.llms import OpenAI; llm = OpenAI(temperature=0)
demo_responses = [
    "예지보전 시스템은 설비 고장을 사전에 예측하여 유지보수 비용을 절감합니다. 진동 센서와 온도 센서 데이터를 분석하여 이상 징후를 탐지합니다.",
    "품질 관리를 위해서는 비전 AI를 활용한 실시간 불량 탐지가 효과적입니다. 긁힘, 오염, 파손 등을 자동으로 분류할 수 있습니다.",
    "스마트 팩토리 구축을 위해서는 IoT 센서, AI, 빅데이터를 통합해야 합니다. 실시간 모니터링과 자동화로 생산 효율을 높일 수 있습니다.",
    "설비 고장 진단에는 Random Forest와 XGBoost 같은 머신러닝 모델이 효과적입니다. 센서 데이터를 학습하여 고장 유형을 분류합니다.",
    "제공된 문서를 기반으로 답변드렸습니다. 추가 질문이 있으시면 말씀해주세요."
]

llm = FakeListLLM(responses=demo_responses)

print("✅ LLM 및 프롬프트 설정 완료")
print("⚠️ 현재 FakeListLLM 사용 중 (데모용)")
print("💡 실제 환경에서는 OpenAI API 사용 권장")

## 🔗 Step 5: ConversationalRetrievalChain 구축

In [ ]:
# ConversationalRetrievalChain 생성
qa_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=retriever,
    memory=memory,
    return_source_documents=True,
    verbose=False,
    combine_docs_chain_kwargs={'prompt': MANUFACTURING_PROMPT}
)

print("✅ ConversationalRetrievalChain 구축 완료")
print("\n🔧 체인 구성 요소:")
print("   1. LLM: FakeListLLM (데모용)")
print("   2. Retriever: ChromaDB (k=3)")
print("   3. Memory: ConversationBufferMemory")
print("   4. Prompt: Manufacturing Domain Customized")
print("   5. Source Documents: 반환 활성화")

## 💬 Step 6: 챗봇 함수 구현

In [ ]:
# 대화 로그 저장용
conversation_logs = []

def chat_with_sources(message, history):
    """
    RAG 기반 챗봇 함수
    
    Args:
        message (str): 사용자 입력 메시지
        history (list): 대화 히스토리 (Gradio 형식)
    
    Returns:
        str: 챗봇 응답 (출처 포함)
    """
    try:
        # 체인 실행
        result = qa_chain({'question': message})
        
        # 응답 및 출처 추출
        answer = result['answer']
        source_docs = result.get('source_documents', [])
        
        # 출처 정보 추가
        if source_docs:
            sources = []
            for idx, doc in enumerate(source_docs, 1):
                source_name = doc.metadata.get('source', 'Unknown')
                category = doc.metadata.get('category', 'N/A')
                sources.append(f"[{idx}] {category} ({source_name})")
            
            sources_text = "\n\n📚 **참고 문서**:\n" + "\n".join(sources)
            full_response = answer + sources_text
        else:
            full_response = answer
        
        # 대화 로그 저장
        conversation_logs.append({
            'timestamp': datetime.now().isoformat(),
            'user': message,
            'assistant': answer,
            'sources': [doc.metadata for doc in source_docs],
            'num_sources': len(source_docs)
        })
        
        return full_response
    
    except Exception as e:
        error_msg = f"⚠️ 오류 발생: {str(e)}"
        conversation_logs.append({
            'timestamp': datetime.now().isoformat(),
            'user': message,
            'assistant': error_msg,
            'sources': [],
            'error': str(e)
        })
        return error_msg

print("✅ 챗봇 함수 구현 완료")
print("\n💡 주요 기능:")
print("   - RAG 기반 응답 생성")
print("   - 출처 문서 표시")
print("   - 대화 로그 자동 저장")
print("   - 에러 핸들링")

## 🎨 Step 7: Gradio UI 구축 및 테스트

In [ ]:
# Gradio ChatInterface 생성
demo = gr.ChatInterface(
    fn=chat_with_sources,
    title="💬 제조 AI 챗봇",
    description="""RAG 기반 제조 분야 전문 챗봇입니다. 예지보전, 품질관리, 스마트팩토리 관련 질문을 해보세요.
    
**예시 질문**:
- 예지보전이 무엇인가요?
- 품질 관리 시스템은 어떻게 작동하나요?
- 스마트 팩토리 구축 방법을 알려주세요
- 설비 고장 진단 방법은?
""",
    examples=[
        "예지보전 시스템의 장점은 무엇인가요?",
        "비전 AI를 활용한 품질 관리 방법을 알려주세요",
        "스마트 팩토리에서 사용하는 기술은?",
        "설비 고장을 예측하는 방법은?"
    ],
    theme="soft",
    retry_btn="🔄 재시도",
    undo_btn="↩️ 실행 취소",
    clear_btn="🗑️ 대화 초기화"
)

print("✅ Gradio UI 구축 완료")
print("\n🎨 UI 구성:")
print("   - ChatInterface: 대화형 인터페이스")
print("   - Examples: 4개 예시 질문 제공")
print("   - Controls: 재시도, 실행취소, 초기화 버튼")
print("\n💡 실행 방법: demo.launch() 호출")
print("⚠️ Jupyter 환경에서는 자동 실행되지 않습니다")

In [ ]:
# 프로그래매틱 테스트 (UI 없이)
print("🧪 챗봇 테스트 시작...\n")

test_questions = [
    "예지보전이 무엇인가요?",
    "품질 관리는 어떻게 하나요?",
    "스마트 팩토리 구축 방법은?"
]

for idx, question in enumerate(test_questions, 1):
    print(f"\n{'='*60}")
    print(f"📝 테스트 {idx}: {question}")
    print(f"{'='*60}")
    
    response = chat_with_sources(question, [])
    print(f"\n🤖 응답:\n{response}")
    print(f"\n{'='*60}")

print(f"\n✅ 테스트 완료 - 총 {len(test_questions)}개 질문 처리")

## 📊 Step 8: 대화 통계 분석

In [ ]:
# 대화 로그 DataFrame 변환
if conversation_logs:
    df_logs = pd.DataFrame(conversation_logs)
    
    print("📊 대화 통계")
    print(f"   - 총 대화 수: {len(df_logs)}")
    print(f"   - 평균 출처 문서 수: {df_logs['num_sources'].mean():.2f}개")
    print(f"   - 에러 발생: {df_logs['assistant'].str.contains('⚠️').sum()}건")
    
    # 대화 길이 분석
    df_logs['user_length'] = df_logs['user'].str.len()
    df_logs['assistant_length'] = df_logs['assistant'].str.len()
    
    print(f"\n📏 메시지 길이 통계")
    print(f"   - 평균 질문 길이: {df_logs['user_length'].mean():.1f}자")
    print(f"   - 평균 응답 길이: {df_logs['assistant_length'].mean():.1f}자")
    
    # 시각화
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # 출처 문서 수 분포
    axes[0].hist(df_logs['num_sources'], bins=range(0, 6), edgecolor='black', alpha=0.7, color='skyblue')
    axes[0].set_xlabel('출처 문서 수', fontsize=12, fontweight='bold')
    axes[0].set_ylabel('빈도', fontsize=12, fontweight='bold')
    axes[0].set_title('📚 출처 문서 수 분포', fontsize=14, fontweight='bold')
    axes[0].grid(alpha=0.3)
    
    # 메시지 길이 비교
    axes[1].bar(['질문', '응답'], 
                [df_logs['user_length'].mean(), df_logs['assistant_length'].mean()],
                color=['lightcoral', 'lightgreen'], edgecolor='black', alpha=0.7)
    axes[1].set_ylabel('평균 길이 (자)', fontsize=12, fontweight='bold')
    axes[1].set_title('📏 메시지 길이 비교', fontsize=14, fontweight='bold')
    axes[1].grid(alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.show()
    
    print("\n✅ 대화 통계 분석 완료")
else:
    print("⚠️ 대화 로그가 없습니다")

## 💾 Step 9: 결과 저장

In [ ]:
# 출력 디렉토리 생성
output_dir = Path('../outputs')
output_dir.mkdir(exist_ok=True)

# 1. 대화 로그 CSV 저장
if conversation_logs:
    df_logs = pd.DataFrame(conversation_logs)
    
    # 출처 정보를 문자열로 변환
    df_logs['sources_str'] = df_logs['sources'].apply(
        lambda x: '; '.join([f"{s.get('category', 'N/A')} ({s.get('source', 'Unknown')})" for s in x])
    )
    
    # 저장용 DataFrame
    df_save = df_logs[['timestamp', 'user', 'assistant', 'num_sources', 'sources_str']].copy()
    df_save.columns = ['시간', '질문', '응답', '출처수', '출처목록']
    
    logs_path = output_dir / 'chatbot_conversation_logs.csv'
    df_save.to_csv(logs_path, index=False, encoding='utf-8-sig')
    print(f"✅ 대화 로그 저장: {logs_path}")
    
    # 2. 통계 요약 저장
    stats_summary = {
        'total_conversations': len(df_logs),
        'avg_sources': float(df_logs['num_sources'].mean()),
        'avg_question_length': float(df_logs['user_length'].mean()),
        'avg_answer_length': float(df_logs['assistant_length'].mean()),
        'error_count': int(df_logs['assistant'].str.contains('⚠️').sum()),
        'timestamp': datetime.now().isoformat()
    }
    
    stats_path = output_dir / 'chatbot_statistics.json'
    with open(stats_path, 'w', encoding='utf-8') as f:
        json.dump(stats_summary, f, ensure_ascii=False, indent=2)
    print(f"✅ 통계 요약 저장: {stats_path}")
    
    # 3. 대화 분석 CSV 저장
    analysis_df = pd.DataFrame({
        '항목': ['총 대화 수', '평균 출처 문서 수', '평균 질문 길이', '평균 응답 길이', '에러 발생'],
        '값': [
            f"{len(df_logs)}건",
            f"{df_logs['num_sources'].mean():.2f}개",
            f"{df_logs['user_length'].mean():.1f}자",
            f"{df_logs['assistant_length'].mean():.1f}자",
            f"{df_logs['assistant'].str.contains('⚠️').sum()}건"
        ]
    })
    
    analysis_path = output_dir / 'chatbot_analysis.csv'
    analysis_df.to_csv(analysis_path, index=False, encoding='utf-8-sig')
    print(f"✅ 대화 분석 저장: {analysis_path}")

# 4. 챗봇 설정 정보 저장
config_info = {
    'model': 'FakeListLLM (Demo)',
    'embeddings': 'jhgan/ko-sroberta-multitask',
    'retriever': {
        'type': 'similarity',
        'k': 3
    },
    'memory': 'ConversationBufferMemory',
    'vector_store': 'ChromaDB',
    'collection_name': 'manufacturing_docs',
    'created_at': datetime.now().isoformat()
}

config_path = output_dir / 'chatbot_config.json'
with open(config_path, 'w', encoding='utf-8') as f:
    json.dump(config_info, f, ensure_ascii=False, indent=2)
print(f"✅ 챗봇 설정 저장: {config_path}")

print(f"\n{'='*60}")
print("✅ 제조 AI 챗봇 구축 완료!")
print(f"{'='*60}")
print(f"📁 저장 위치: {output_dir.absolute()}")
print(f"📊 저장 파일:")
print(f"   1. chatbot_conversation_logs.csv - 대화 로그")
print(f"   2. chatbot_statistics.json - 통계 요약")
print(f"   3. chatbot_analysis.csv - 대화 분석")
print(f"   4. chatbot_config.json - 챗봇 설정")

---
## 🎯 학습 정리

### ✅ 완료한 내용
1. ✅ **RAG 시스템 통합**: ChromaDB 벡터 스토어 활용
2. ✅ **대화 메모리 관리**: ConversationBufferMemory로 문맥 유지
3. ✅ **LangChain 체인 구축**: ConversationalRetrievalChain 생성
4. ✅ **제조 도메인 맞춤화**: 전문 프롬프트 템플릿 적용
5. ✅ **Gradio UI 구축**: 사용자 친화적 대화형 인터페이스
6. ✅ **출처 표시 기능**: 참고 문서 자동 추출 및 표시
7. ✅ **대화 로그 저장**: 분석을 위한 자동 로깅
8. ✅ **통계 분석**: 대화 패턴 및 성능 분석

### 💡 핵심 인사이트
- **RAG의 강점**: LLM의 한계(지식 cutoff, hallucination)를 벡터 검색으로 보완
- **메모리 관리**: ConversationBufferMemory로 다중 턴 대화에서 문맥 유지
- **출처 추적**: 답변의 신뢰성 향상을 위한 출처 문서 표시 필수
- **도메인 맞춤화**: 제조 분야 전문 프롬프트로 응답 품질 향상
- **UI/UX**: Gradio로 비개발자도 쉽게 사용 가능한 인터페이스 구현

### 📚 실무 적용 방안
1. **설비 고장 진단 챗봇**: 과거 고장 이력 DB 기반 원인 분석
2. **품질 관리 가이드**: 불량 유형별 대응 매뉴얼 검색
3. **예지보전 컨설팅**: 센서 데이터 해석 및 유지보수 권장사항
4. **스마트 팩토리 Q&A**: 도입 사례 및 기술 스택 추천
5. **신입 교육**: 제조 공정 및 AI 활용 방법 교육 도구

### 🔧 개선 방향
- **LLM 업그레이드**: OpenAI GPT-4 또는 Claude API 연동
- **고급 검색**: Hybrid Search (BM25 + Semantic) 적용
- **다국어 지원**: 영어, 중국어 등 다국어 문서 처리
- **음성 인터페이스**: 음성 입력/출력 추가 (현장 작업자용)
- **피드백 루프**: 사용자 평가 수집 및 모델 개선

### 📖 참고 자료
- [LangChain 공식 문서](https://python.langchain.com/docs/)
- [Gradio 튜토리얼](https://www.gradio.app/docs/)
- [ChromaDB 가이드](https://docs.trychroma.com/)
- [RAG 개념 설명](https://huggingface.co/docs/transformers/model_doc/rag)

---

*제조AI 교육 v12 Enhanced | Part 3-1 | 2025.02*  
*문의: AI 교육팀 | 난이도: ⭐⭐⭐⭐*